In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

RAW_PR_PATH = "/Volumes/rearc_bls/bronze/raw_landing/pr"
RAW_POP_PATH = "/Volumes/rearc_bls/bronze/raw_landing/population/population.json"


def _read_bls_flat_file(filename: str):
    """BLS flat files are tab-separated and pad column headers with
    whitespace (e.g. 'series_id        '), so every read goes through
    this helper to normalize column names before anything downstream
    references them by name."""
    raw = (
        spark.read.option("header", True).option("sep", "\t")
        .csv(f"{RAW_PR_PATH}/{filename}")
    )
    return raw.toDF(*[c.strip() for c in raw.columns])


# ---------------------------------------------------------------------------
# Core series data
# ---------------------------------------------------------------------------

@dp.table(name="rearc_bls.bronze.bronze_bls_pr", comment="Raw BLS productivity series values")
@dp.expect_or_drop("valid_series_id", "series_id IS NOT NULL")
def bronze_bls_pr():
    raw = _read_bls_flat_file("pr.data.0.Current")
    return raw.select(
        F.trim(F.col("series_id")).alias("series_id"),
        F.col("year").cast("int").alias("year"),
        F.trim(F.col("period")).alias("period"),
        F.col("value").cast("double").alias("value"),
        F.trim(F.col("footnote_codes")).alias("footnote_codes"),
    )


# ---------------------------------------------------------------------------
# Series metadata + code lookup tables (needed to build human-readable labels)
# ---------------------------------------------------------------------------

@dp.table(name="rearc_bls.bronze.bronze_bls_series", comment="BLS series metadata (codes only, no title column)")
def bronze_bls_series():
    return _read_bls_flat_file("pr.series")


@dp.table(name="rearc_bls.bronze.bronze_bls_sector", comment="BLS sector code lookup")
def bronze_bls_sector():
    return _read_bls_flat_file("pr.sector")


@dp.table(name="rearc_bls.bronze.bronze_bls_class", comment="BLS class code lookup")
def bronze_bls_class():
    return _read_bls_flat_file("pr.class")


@dp.table(name="rearc_bls.bronze.bronze_bls_measure", comment="BLS measure code lookup")
def bronze_bls_measure():
    return _read_bls_flat_file("pr.measure")


@dp.table(name="rearc_bls.bronze.bronze_bls_duration", comment="BLS duration code lookup")
def bronze_bls_duration():
    return _read_bls_flat_file("pr.duration")


# ---------------------------------------------------------------------------
# Population (DataUSA API JSON)
# ---------------------------------------------------------------------------

@dp.table(name="rearc_bls.bronze.bronze_population", comment="Raw population API response exploded to one row per record")
def bronze_population():
    df = spark.read.option("multiLine", True).json(RAW_POP_PATH)
    exploded = df.select(F.explode("data").alias("record")).select("record.*")
    # DataUSA's API returns fields like "Nation ID" with spaces, which Delta
    # rejects as column names — replace spaces (and other invalid chars) with underscores
    clean_cols = [c.replace(" ", "_") for c in exploded.columns]
    return exploded.toDF(*clean_cols)